In [9]:
import streamlit as st
from sentence_transformers import SentenceTransformer, util
import json
import numpy as np

# Load clustered CQs (you can switch between different clustering outputs)
with open("competency_questions_output/clustered_CQs_hdbscan.json", "r", encoding="utf-8") as f:
    clustered_cqs = json.load(f)

# Flatten and prepare CQ list with source cluster
cq_list = []
for cluster_id, cqs in clustered_cqs.items():
    for cq in cqs:
        cq_list.append({"cluster": cluster_id, "text": cq})

# Load model and encode all CQs
@st.cache_resource(show_spinner=False)
def load_model_and_embeddings():
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    embeddings = model.encode([cq["text"] for cq in cq_list], convert_to_tensor=True)
    return model, embeddings

model, cq_embeddings = load_model_and_embeddings()

# Streamlit UI
st.title("🔍 Juristisches Frage-Dashboard")
st.markdown("Stelle eine juristische Frage und finde passende Kompetenzfragen aus dem Ontologie-Cluster.")

user_input = st.text_input("Ihre Frage eingeben:")

if user_input:
    user_embedding = model.encode(user_input, convert_to_tensor=True)
    cos_scores = util.cos_sim(user_embedding, cq_embeddings)[0]
    top_k = min(5, len(cq_list))
    top_results = np.argpartition(-cos_scores, range(top_k))[:top_k]

    st.subheader("Ähnliche Kompetenzfragen")
    for idx in top_results:
        score = float(cos_scores[idx])
        cq = cq_list[idx]
        st.markdown(f"**{cq['text']}**")
        st.caption(f"📂 Cluster: {cq['cluster']} | Ähnlichkeit: {score:.2f}")


ModuleNotFoundError: No module named 'streamlit'